# Beyond the Data Store Graphical User Interface (GUI): Hands-On with pyBIS
**Beginner friendly workshop | 210 minute**

## Introduction

### Welcome!

In the previous Lab Notebook tutorial, you learned how to organize and document research data using the BAM Data Store graphical user interface (GUI).

In this workshop, you will explore another way of interacting with the Data Store: **pyBIS**, a Python library that allows you to access and manage openBIS data through code.

**No previous experience with Python, Jupyter Notebooks, or pyBIS is required.** We will guide you through the exercises step by step.

### What will you learn?

By the end of this workshop, you will be able to:

1. **Retrieve research data:** Find and access your projects, collections, and experimental steps using pyBIS.
2. **Modify metadata:** Update the description of a practice record and verify your changes in the Data Store GUI.
3. **Automate repetitive tasks:** Retrieve information from multiple records and export it as a table (CSV file).

### What data will we use?

This workshop builds on the [BAM Lab Notebook tutorial](https://bamresearch.github.io/openBIS-data-store-tutorials/lab-notebook/) and is adapted from `lab-notebook-pybis.ipynb`.

Rather than repeating the entire tutorial, we will work with a small example research workflow consisting of:

- A project called `Material_Synthesis`.
- Three collections: `Synthesis`, `Parameter 1`, and `Parameter 2`.
- A few Experimental Steps representing research activities.

You will either reuse data from the previous GUI tutorial or work with example data prepared for this workshop.

### Before you begin

- **Use only your assigned training Space.** This keeps your exercises separate from other participants' work.
- **Keep the original tutorial data unchanged.** We will create a dedicated practice record (`Dummy_Object`) for the metadata editing exercise.
- **Your computer should already be prepared.** VS Code, Python, pyBIS, and the required environment should have been installed following the separate setup instructions. No terminal commands or additional installation should be necessary during the workshop.

---

## Glossary — Understanding openBIS and pyBIS terminology

The BAM Data Store is based on **openBIS**. Some concepts have different names in the **Data Store graphical user interface (GUI)** and **pyBIS (Python interface)**.

This glossary explains the most important terms used throughout the workshop.

### Organization of research data

- **Space:** The highest organizational level in openBIS, used to organize research data and manage access permissions, for example, by separating working groups. You have permission to access the content of your assigned Lab Notebook Space, but you may not have access to other Spaces unless additional permissions are granted.

- **Project:** A Space can contain multiple Projects. Projects organize related research activities and their associated Collections.

- **Collection:** A Project can contain multiple Collections, Collections group multiple objects. In the Data Store is recommended to group similar objects.

- **Object:** A Collection can contain multiple Objects. Objects represent research entities such as an instrument or an experimental step, and have metadata properties.

- **Dataset:** A Dataset contains the actual research data files. In the Data Store, Datasets are typically attached to Objects that describe how the data was generated, often an Object of type `EXPERIMENTAL_STEP`.

### Attributes and properties

- **Attributes:** System-defined fields that describe an openBIS entity, such as `code`, `permId`, `identifier`, `type`, and `registrationDate`.
  Attributes are available for common entity categories, including Spaces, Projects, Collections, Objects, and Datasets.

- **Properties:** Type-specific metadata fields used to describe research objects in a structured way.
  Properties are defined through *Property Types* and assigned to *Entity Types* during the masterdata definition process.

  Properties are supported in openBIS for **Collections, Objects, and Datasets**. Spaces and Projects have system-defined attributes but do not contain type-specific properties.

  **Example:** An experimental step might have properties describing the temperature, duration, or responsible person.

### Entity types and property types

- **Entity Type:** Defines which metadata properties are available for a particular category of research record.

  In openBIS, Entity Types include:

  - **Collection Type:** Defines the properties available for a Collection.
  - **Object Type:** Defines the properties available for an Object.
  - **Dataset Type:** Defines the properties available for a Dataset.

  In the **BAM Data Store**, only the Collection type **Collection** and the Dataset type **Raw Data** are used.  However, different **Object Types** can be defined by users and made available after approval by the **BAM Data Model Task Force**.

---


## Research Data Hierarchy — pyBIS Workshop Example

In the BAM Data Store, research data is organized hierarchically:

**Space → Project → Collection → Object → Dataset**


The diagram below shows the entities we will use during the hands-on challenges (1–5).

```text
SPACE: YOUR_LAB_NOTEBOOK_SPACE
│
└── PROJECT: Material_Synthesis
    │
    ├── COLLECTION: Parameter_1
    │   │
    │   ├── OBJECT: Macrostructure_Characterization
    │   │   
    │   │
    │   └── OBJECT: Macrostructure_Characterization_TS
    │      
    │
    ├── COLLECTION: Parameter_2
    │   │
    │   └── OBJECT: Microstructure_Characterization
    │       
    │
    └── COLLECTION: Synthesis
        │
        ├── OBJECT: Material_Synthesis
        │   
        │
        └── OBJECT: Dummy_Object
            └── DATASET(S):  Zip file; of types RAW_DATA and ELN_PREVIEW
```

### Understanding the hierarchy

- **Space:** Your assigned Lab Notebook Space. It determines where you work and which research records you can access.
- **Project:** `Material_Synthesis` organizes the research activities used in this workshop.
- **Collections:** `Parameter_1`, `Parameter_2`, and `Synthesis` group related experimental activities.
- **Objects:** Represent individual research activities, such as material synthesis or characterization.
- **Datasets:** Contain the actual research data files associated with an Object.

### Important for the hands-on exercises

- We will **retrieve and explore** existing research records using pyBIS.
- We will use `Dummy_Object` as a **practice Object** for modifying metadata.
- We will **not modify the original research Objects** created during the GUI tutorial.
- We will **upload data** of different types and formats to the `Dummy_Object`
- we will **retrieve objects and export metadata** 


## Step 0 - Check your workshop environment

Before we start working with the Data Store, let's make sure your Python environment is working correctly.

You should have already installed VS Code, Python, pyBIS, and the required libraries following the separate setup instructions.

**What to do:**

1. Open this notebook in VS Code
2. Make sure you have selected the Python environment prepared during setup <code>pybis(Python 3.14.8) or pybis(Python 3.14.10)</code>
3. Run the following code cell using the **▶ Run** button or **Shift + Enter**.
4. Check that the cell finishes without an error.

**Expected result**: You should see a confirmation message "Environment check passed" indicating that Python and the required libraries are available.


If you see an error, ask an instructor for help before continuing.


In [ ]:
import sys
from pathlib import Path

expected_env_name = "pybis"
active_prefix = Path(sys.prefix)

is_conda_env = (active_prefix / "conda-meta").is_dir()
is_expected_env = active_prefix.name.casefold() == expected_env_name.casefold()
is_python_314 = sys.version_info[:2] == (3, 14)

print("Python executable:", sys.executable)
print("Environment:", active_prefix)
print("Conda environment:", is_conda_env)
print("Expected environment (pybis):", is_expected_env)
print("Python version:", sys.version.split()[0])

if not (is_conda_env and is_expected_env and is_python_314):
    raise RuntimeError("Select the pybis Conda environment with Python 3.14, then rerun this cell.")

print("Environment check passed.")

In [ ]:
from importlib.metadata import PackageNotFoundError, version

print("Python executable:", sys.executable)

for package in ("pybis", "ipykernel"):
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError as error:
        raise RuntimeError(f"{package} is not installed in this environment.") from error

print("Both packages are installed and import successfully.")

In [ ]:
#commented libraries are listed in case some users experience issues
#you don't need to install them if you got the message "Environment check passed" in the previous cell.

# %pip install pybis
# %pip install ipykernel
%pip install pandas
# # %pip install jupyterlab


## Part A — Connect and prepare
### 1. Connect to openBIS
1. Open the BAM Data Store playground in your browser and log in.
2. In **Tools → User Profile**, copy your openBIS session token. 
3. Run the next cell with **Shift + Enter**. Paste the token into the input field display in the upper part of the VS code menu. **Never paste a token into a saved code cell or share it.**

In [ ]:
from getpass import getpass
import pandas as pd
from pybis import Openbis
from helper import *

o = Openbis("https://playground.datastore.bam.de/")
o.set_token(getpass("Paste your openBIS session token (hidden): "))
print("pyBIS connection configured. Next, select your Space.")

## Step 1: Select your own Space

Each participant has an assigned a Space in the BAM Data Store instances.

**Your task:**

1. Run the first code cell to display all Spaces you have permission to access.
2. Find your assigned Space in the list.
3. Copy its **Space code**.
4. In the second code cell, replace `"your_space_code"` with your assigned Space code.
5. Run the second code cell to select your Space.

**Expected result:** The identifier of your selected Space is displayed.

In [ ]:
# Display all Spaces you have permission to access
# Find your assigned training Space in the list, this shows your division numer and User name separated by underscore (e.g., FB_EBAMUSER)

display(o.get_spaces())

In [ ]:
#**TO DO AN: review and uncomment or delete**

# Select your assigned training Space

# YOUR TURN: Replace "your_space_code" with your assigned Space code. For example "X.1_EBAMUSER"
# Keep the quotation marks around your Space code


# space_code ="X.1_CBAMUSER"

In [ ]:
#**TO DO AN: review and uncomment or delete**

# #Retrieve your Space from the BAM Data Store
# myspace = o.get_space(space_code)

In [ ]:
# Display the identifier of your selected Space
# This confirms which Space you are working with

#print("Selected Space:", myspace.identifier)

In [ ]:
# YOUR TURN: Replace 'your_space_code' by your space code. For example 'X.1_CBAMUSER' (with quotation marks). 
myspace = o.get_space('your_space_code')



## Step 2: Check that the project, collections, and experimental steps needed for workshop exercises are registered in your Lab Notebook

**Path A:** You completed the GUI tutorial. The notebook looks for the existing `Material_Synthesis` project and its collections.

**Path B:** You did not complete the GUI tutorial. The notebook creates **only missing** minimal structures: a project, three collections and a few demonstration experimental steps.

Run the same cells for either path. **No need to decide manually.** The checks are based on actual objects.

In [ ]:
PROJECT_CODE = "Material_Synthesis" # Name of the project

object_codes = {
    'Macrostructure Characterization': 'Parameter 1',
    'Macrostructure Characterization TS': 'Parameter 1',
    'Microstructure Characterization': 'Parameter 2',
    'Material Synthesis': 'Synthesis'
}
collection_codes = ["Parameter 1", "Parameter 2", "Synthesis"]
project_missing_flag = False
missing_collections = []
missing_objects = []
path = 'B'#default value


try:#try except for catching no project error
    project = myspace.get_project(project_code=PROJECT_CODE)
except Exception:
    print('Project Material Synthesis Missing')
    project_missing_flag = True
    missing_collections = collection_codes.copy()

if not project_missing_flag:#if project exists
    project_collections = project.get_collections()
    collections_by_name = {
        collection.props.get('$name'): collection
        for collection in project_collections
    }
    missing_collections = [
        name for name in collection_codes if name not in collections_by_name
    ]
    for template_name, collection_name in object_codes.items():
        collection = collections_by_name.get(collection_name)
        if collection is None:
            continue
        existing_object_names = {
            object.props.get('$name') for object in collection.get_objects()
        }
        if template_name not in existing_object_names:
            missing_objects.append((template_name, collection_name))

if not project_missing_flag and not missing_collections and not missing_objects:
    print('Project, collections, and expected objects exist')
    path = 'A'

print('Missing collections:', missing_collections)
print('Missing objects:', missing_objects)
print('Path =', path)

In [ ]:
# If Path B, create only the entities that are required for this notebook exercises.
if path == 'B':
    if project_missing_flag:
        project = myspace.new_project(
            code=PROJECT_CODE,
            description='This tutorial project demonstrates a simplified workflow for the synthesis and characterization of a material within openBIS. The experimental descriptions, metadata, and datasets were generated using AI and represent synthetic example data created solely for training and demonstration purposes. Although the content has been reviewed for consistency and usability within the tutorial, it has not been validated for scientific accuracy and should not be considered scientifically reliable or reproducible.'
        )
        project.save()
        collections_to_create = collection_codes
    else:
        collections_to_create = missing_collections

    for name in collections_to_create:
        collection = o.new_collection(
            code=auto_generated_code(project),
            type='Collection',
            project=project.identifier,
            props={'$name': name}
        )
        collection.save()

    collections_by_name = {
        collection.props.get('$name'): collection
        for collection in project.get_collections()
    }

    for template_name, collection_name in object_codes.items():
        collection = collections_by_name[collection_name]
        existing_object_names = {
            object.props.get('$name') for object in collection.get_objects()
        }
        if template_name in existing_object_names:
            continue

        if template_name == 'Macrostructure Characterization TS':
            object_props = templates['Macrostructure Characterization'].copy()
        else:
            object_props = templates[template_name].copy()
        object_props['$name'] = template_name

        template_object = o.new_object(
            type='EXPERIMENTAL_STEP',
            collection=collection,
            props=object_props
        )
        template_object.save()

**GUI checkpoint:** Open the **Material Synthesis** Project in the BAM Data Store.

Check that you can find the following three Collections:

- **Parameter 1**
- **Parameter 2**
- **Synthesis**

Then, explore the Collections and check that you can find the following Experimental Steps (Objects):

- `Macrostructure Characterization`
- `Macrostructure Characterization TS`
- `Microstructure Characterization`
- `Material Synthesis`

**Expected result:** You can locate all three Collections and the four Experimental Steps listed above.

If everything is visible, you are ready to start the hands-on exercises!

---
## Step 3: Hands-on challenges
### Challenge 1 — Retrieve a registered object: experimental step (30 minutes)

In the Data Store web interface, you can navigate through projects and collections to find an experimental step.

You can perform the same operation using **pyBIS**. This is useful when you want to retrieve information about registered Objects using Python instead of searching for them manually in the web interface.

**In this exercise...**

You will use pyBIS to find a Collection in your Project and retrieve an Experimental Step stored in that Collection.

**What the code does:**

1. Retrieves the collections in your project.
2. Finds the collection with the specified name.
3. Retrieves the Objects stored in that collection.
4. Displays the name and identifier of one experimental step.

An **identifier** is a unique reference used by openBIS to distinguish one object from another.

**Your task:** 
1. Run the first code cell to retrieve an Object from the `Parameter 2` Collection.
2. Look at the displayed Object name and identifier.
3. In the second code cell, change the Collection name from `"Parameter 2"` to `"Synthesis"`.
4. Run the modified code cell.
5. Compare the results.

**Expected result:** 

The first code cell displays the name and identifier of an Object from the `Parameter 2` Collection.

After changing the Collection name, the second code cell displays the name and identifier of an Object from the `Synthesis` Collection.

**GUI Checkpoint:** 
Open the **Material Synthesis** Project in the BAM Data Store web interface.

Can you find the same Experimental Step in the `Synthesis` Collection?

**Important notes:**

- Collection names must match exactly, including capitalization and spaces.
- Python starts counting at `0`, so `objects[0]` refers to the first Object in a list.
- This exercise only retrieves information. It does **not** modify any data in the Data Store


In [ ]:
# Display the identifier of the project we are exploring
print("Project:", project.identifier)

# Find the collection named "Parameter 2"
# Empty Variables to hold the collection and its objects
collection = None
objects = None

for collection in project.get_collections():
    # Check the name of each collection and match it with the name we are looking for
    if collection.props.all()["$name"] == 'Parameter 2':

        #Store the objects in the variable for the next step
        objects = collection.get_objects()
        break

# Display the selected collection and its first object
# The object name is the human-readable name; the identifier uniquely identifies it
print('Collection: ',collection.props.all()["$name"])
print('Object Name: ',objects[0].props.all()['$name'])
print('Object Identifier: ',objects[0].identifier)

#**Expected Output**
#code cell displays the name and identifier of an Object from the `Parameter 2` Collection.

### Your task: replace the collection name from `Parameter 2` to `Synthesis`.

In [ ]:
# Create an empty variable to store the Objects
objects = None

 # Check the name of all collections within the project
for collection in project.get_collections():
   
    # Check whether the Collection name matches the one we want
    # YOUR TURN: Change 'Parameter 2' to 'Synthesis'
    if collection.props.all()["$name"] == 'Parameter 2':

        # Retrieve all Objects from the selected Collection
        # Store them in the variable "objects"
        objects = collection.get_objects()
        # Stop the loop because we found the Collection
        break

# Display the name and identifier of the first Object
# Python starts counting at 0, so objects[0] means the first Object

print('Object Name: ',objects[0].props.all()['$name'])
print('Object Identifier: ',objects[0].identifier)

#**Expected Output**
#After changing the Collection name, the second code cell displays the name and identifier of an Object from the `Synthesis` Collection.

**Check:** Can you find the same experimental step in the Data Store web interface?

**Expected result:** You should see the name and identifier of the experimental step in the Synthesis collection.


### Challenge 2 — Explore your research data as a table (30 minutes)

When a project contains many experimental steps, inspecting them one by one can be time-consuming.

Python can retrieve information from multiple object and organize it into a table.

In this exercise, we use **pandas**, a Python library for working with tabular data.

**The table contains four columns:**

- `collection`: The collection containing the experimental step.
- `identifier`: The unique identifier of the record.
- `name`: The human-readable name of the experimental step.
- `type`: The openBIS object type.

You do not need to understand every line of code. Focus on the table produced by the code.

**Your task:** Change `collection_to_find` from `Parameter 1` to `Synthesis`.

**Expected result:** The displayed table contains only experimental steps from the selected collection.

**Think about it:** How could filtering help you find experimental records in a project containing hundreds of steps?


In [ ]:
# Create an empty list to collect information about our objects
# Each object will later become one row in our table
rows = []

# Select the two collections we want to explore
# We look for collections named "Synthesis" and "Parameter 1"
# The next() function returns the first collection matching the name
for collection_name, collection in [
    ("Synthesis", next(col for col in project.get_collections()if col.props["$name"] == "Synthesis")),
    ("Parameter 1", next(col for col in project.get_collections()if col.props["$name"] == "Parameter 1"))
]:

    # Retrieve all objects in the selected collection
    for obj in collection.get_objects():

        # Extract information from each object
        # Store the information as a dictionary (key-value pairs)
        # Each dictionary will become one row in the table
        rows.append({
            "collection": collection_name,   # Collection name
            "identifier": obj.identifier,     # Unique object identifier
            "name": obj.props.get("$name"),   # Human-readable object name
            "type": obj.type,                 # Object type in openBIS
        })

# Convert the collected information into a pandas DataFrame
# A DataFrame is a table with rows and columns
objects_table = pd.DataFrame(rows,columns=["collection", "identifier", "name", "type"])

# Display the table in the notebook
display(objects_table)

# Count and display how many objects were found
print("Total objects:", len(objects_table))

#**Expected Output:** The displayed table contains only experimental steps from the selected collection(s).


### Your task: Change `collection_to_find` from `Parameter 1` to `Synthesis`.


In [ ]:
# Choose the Collection you want to explore
# YOUR TURN: Change "Parameter 1" to "Synthesis"
collection_to_find = "Parameter 1" 

# Filter the table to show only Objects from the selected Collection
# Python checks which rows have a matching value in the "collection" column
filtered = objects_table[objects_table["collection"] == collection_to_find]

# Display the filtered table
display(filtered)

# Count how many Objects were found
# len() returns the number of rows in the filtered table
print("Matching objects:", len(filtered))

# **Expected Output:** The displayed table contains only experimental steps from the Synthesis collection.

**Hint:** The Collection name must match exactly, including capitalization and spaces. 
**Note:** If the name does not match any Collection in the table, the result will be an empty table.

### Challenge 3 — Modify metadata of an experimental step

#### Why do this?

Research metadata describes how an experiment was performed, for example, the materials, equipment, or experimental conditions used.

Sometimes, this information needs to be corrected or updated. Instead of editing each record manually in the Data Store graphical user interface (GUI), you can use **pyBIS** to update metadata.

**Important:** Retrieving information with pyBIS does not change anything in the Data Store. However, saving a modified object updates the information stored in openBIS.

**In this exercise** you will update the description of a practice Object called `Dummy_Object`, located in the `Synthesis` Collection.



You will perform three steps:

1. **Register** practice object 
2. **Inspect:** Retrieve the Object and display its current description.
3. **Prepare a change:** Enter a new description and preview the change without saving it.
4. **Save and verify:** Save the new description, retrieve the Object again, and verify the change in the Data Store GUI.

#### Your task

1. Run the first code cell to retrieve `Dummy_Object` and inspect its current description.
2. Change the value of `new_description` to a short description of your choice.
3. Run the preview cell to check the proposed change.
4. Change `SAVE_CHANGE` from `False` to `True` when you are ready to save.
5. Run the save cell.
6. Open the Data Store GUI and find `Dummy_Object` in the `Synthesis` Collection.
7. Check whether the description has been updated.


#### Expected result

- The notebook displays the original description of `Dummy_Object`.
- You can preview the proposed description before saving.
- After saving, the updated description is stored in openBIS.
- The same description is visible in the Data Store GUI.

#### Important notes

- **Only modify `Dummy_Object`.** Do not change the original research Objects created during the previous GUI tutorial as you need them for follow-up tutorials.
- The `.save()` method sends your changes to openBIS.
- Changes are not saved to openBIS until `.save()` is executed.
- If you run the save cell again, you may overwrite the description with the same or a different value.

**Think about it:** Why is it useful to inspect the existing metadata before modifying it?

In [ ]:
# Find the Synthesis collection in our project
collection = next(col for col in project.get_collections()if col.props["$name"] == "Synthesis")

# Look for our practice Object
practice_name = "Dummy_Object"

matching_objects = [
    obj for obj in collection.get_objects()
    if obj.props.get("$name") == practice_name
]

# Reuse the practice Object if it already exists
# This prevents creating a duplicate when the cell is run again
if matching_objects:
    object = matching_objects[0]
    print("Existing practice Object found.")

else:
    # Create a new practice Object in the Synthesis collection
    object = o.new_object(
        type="EXPERIMENTAL_STEP",
        collection=collection,
        props={"$name": practice_name}
    )

    # Register the new Object in openBIS
    object.save()
    print("New practice Object created.")

# Define the metadata property we want to edit
PROPERTY = "experimental_step.experimental_description"

# Display the current metadata
print("Object Name:", object.props.get("$name"))
print("Identifier:", object.identifier)
print("Current description:", object.props.get(PROPERTY))

# Expected Output: The practice Object is sucessfully created and its Name and Identifier are displayed, along with the current description (none).

### Your task: Change the `Description` for `Dummy_Object` and save it.

In [ ]:
# Enter a new description for the practice Object
# YOUR TURN: Change the text between the quotation marks
new_description = "I updated this training record using pyBIS!"

# Preview the proposed change
# Nothing is saved to openBIS at this stage
print("Current description:", object.props.get(PROPERTY))
print("Proposed description:", new_description)
print("No changes have been saved yet.")

# Expectd Output: The current and proposed descriptions are displayed, but the change is not yet saved in openBIS.

In [ ]:
# Decide whether to save the change
# YOUR TURN: Change False to True when you are ready
SAVE_CHANGE = False  # <-- Change this to True to save the change

# Save the new description only if confirmed
if SAVE_CHANGE:

    # Update the property in the Python Object
    object.props[PROPERTY] = new_description

    # Send the change to openBIS
    object.save()

    print("Changes saved successfully!")

else:
    # The original description remains unchanged
    print("Preview only. No changes saved.")

    # Expected Output: The proposed description is saved in openBIS if SAVE_CHANGE is set to True.

In [ ]:
# Retrieve the practice Object again from openBIS
# This allows us to we check the description saved on the server

verified_object = o.get_object(object.identifier)

# Display the description stored in openBIS
print("Object Name:", verified_object.props.get("$name"))
print("Identifier:", verified_object.identifier)
print("Description in openBIS:", verified_object.props.get(PROPERTY))

# Compare the stored description with our proposed change
if verified_object.props.get(PROPERTY) == new_description:
    print("Verification successful: Descriptions match!")
else:
    print("The stored description differs from the proposed change.")

Object Name: Dummy_Object
Identifier: /X.1_CBAMUSER/MATERIAL_SYNTHESIS/EXP8919
Description in openBIS: I updated this training record using pyBIS!
Verification successful: Descriptions match!


**GUI checkpoint:**
- Open the Data Store GUI, refresh the page, and find `Dummy_Object` in the `Synthesis` Collection.
- Check whether the description has been updated.

### Challenge 4 - Upload datasets to the Data Store using pyBIS  (30 minutes)

#### Why do this?

Research experiments often generate many data files from diverse formats, such as images, measurements, or documentation.

In the Data Store graphical user interface (GUI), you can upload these files manually and attach them to an experimental step (object).

With **pyBIS**, you can perform the same task using Python. This is particularly useful when you need to upload multiple files or automate repetitive data uploads.

#### In this exercise...

You will upload three datasets, each with a different **Dataset Type**, to the `Dummy_Object` in the `Synthesis` Collection. 

You will perform four steps using pyBIS:

1. **Prepare the files:** Extract a ZIP archive containing example research data.
2. **Upload raw data:** Register the extracted files as one `RAW_DATA` dataset.
3. **Upload a preview:** Register an example image as an `ELN_PREVIEW` dataset.
4. **Upload an Atrachment:** Register a text file as an `ATTACHMENT` dataset.

**What are Dataset Types?**

`RAW_DATA`, `ELN_PREVIEW`and `ATTACHMENT` are Dataset Types in openBIS.
A Dataset Type determines which metadata properties can be assigned to a dataset and may influece how the data is displayed and previewed in openBIS. 

**Dataset Types are not file formats** For example, `png`, `txt`and `zip` are file formats, while `RAW_DATA`  is an openBIS-Dataset Type.

#### Before you begin — Check your files

**Note:** The Instructors will provide a `datasets` folder containing two individual files and one zip archive.  Place the `datasets` folder in the same directory as this notebook.

Your folder structure should look like this:

```text
WORKSHOP_FOLDER/
│
├── pybis_workshop_exercises.ipynb
│
└── datasets/
    ├── Parameter1_Macroscopic.zip
    ├── Synthetic-image1.png
    └── README-Synthetic-image1.txt
```

#### Your task

1. Run the first code cell to extract the ZIP archive.
2. Upload the extracted files as a `RAW_DATA` dataset.
3. Upload `Synthetic-image1.png` as an `ELN_PREVIEW` dataset.
4. In the final upload cell, change the dataset type from `ELN_PREVIEW` to `ATTACHMENT`.
5. Run the final cell to upload the README text ile.
6. Open the Data Store GUI and verify that all three datasets are attached to `Dummy_Object`.

#### Expected result

The practice Object `Dummy_Object` should contain three datasets:

| Dataset name | Dataset type | Content |
|---|---|---|
| Parameter 1 raw | `RAW_DATA` | Multiple example files extracted from the ZIP archive |
| Synthetic image1 | `ELN_PREVIEW` | An example image |
| README Synthetic image1 | `ATTACHMENT` | A text file explaining the example image |

#### Important notes

- **Only upload datasets to `Dummy_Object`.** Do not modify the original Objects from the GUI tutorial.
- A **Dataset Type** describes how a dataset is categorized in openBIS. It is different from a file format.
- The `.save()` method registers the dataset and uploads the associated files to openBIS.
- **Running an upload cell code creates another dataset contaitning the same files**. Run each upload cell only once.
- A ZIP archive is a compressed file that can contain multiple files and folders.
- In this exercise, we extract the ZIP archive to upload its contents as individual files within one dataset. ZIP archives can also be uploaded to openBIS without extraction.

**Think about it:** Why might it be useful to upload a complete folder of experimental files using pyBIS instead of uploading each file manually through the GUI?


In [ ]:
# Import the Python libraries we need
# zipfile allows Python to open and extract ZIP archives
# Path helps us work with files and folders

import zipfile
from pathlib import Path

# Define the folder containing our example datasets
# The "datasets" folder must be in the same directory as the notebook

datasets_folder = Path("datasets")

# Define  the location of the ZIP archive
# The / symbol joins the folder path and the filename
zip_path = datasets_folder / "Parameter1_Macroscopic.zip"

# Define the folder where the extracted files will be stored
# This folder will be created inside "datasets"
extract_dir = datasets_folder / "Parameter1_Macroscopic"

# Check whether  the ZIP archive exists
# If the file is missing, display an error message and stop
if not zip_path.is_file():
    raise FileNotFoundError(
        f"File not found: {zip_path}\n"
        "Check that the datasets folder contains the ZIP archive."
    )

# Create the destination folder if it does not exist
# If the folder already exists, Python will continue without an error
extract_dir.mkdir(parents=True, exist_ok=True)

# Open the ZIP archive and extract its contents
# "r" means read mode: we are reading the ZIP archive
# The extracted files are saved in the destination folder
with zipfile.ZipFile(zip_path, "r") as zip_file:
    zip_file.extractall(extract_dir)

# Confirm that the extraction has finished
print("ZIP archive extracted successfully!")
print("Extracted files are stored in:", extract_dir)

# Display the names of the extracted files
# The loop checks the destination folder and its subfolders
# Only files are displayed, not folders
for file in extract_dir.rglob("*"):
    if file.is_file():
        print("-", file.name)

#**Expected Output:** The names of the extracted files are displayed, confirming that the ZIP archive was successfully extracted.

**TO DO FOR AN: Check that the zip files does not contain the same .Synthetic image 1.png four times**
**Confirm whether the same image is not uploaded 4 times at the end of challenge 4 and correct if required**

**TO DO:  DELETE this markdown cell, ONCE QUESTION IS RESOLVED BEFORE PUBLICATION**

**TO DO FOR AN**: Review idea, test code and decide on code cell below (Find the Synthesis colelction)Is this useful to TROUBLESHOOT FILE-LOCATION PROBLEMS?**
**should the code be kept, in commented #form?**

**The folder must be in the same directory as the notebook.

is appropriate as a workshop instruction, but technically Python resolves "datasets" relative to its current working directory, which is not always the notebook's directory.

If VS Code starts the notebook in the expected workshop folder, your code works as intended.

To help participants troubleshoot file-location problems, you could optionally display the resolved path:


In [ ]:
**TO DO AN**

#**ACTION NEEDED for AN. Review, TEST AND change if required**

#**the python variable is with capitalize D: Dummy_Object. The variable should be with non-capitalize d: dummy_object**
#**ACTION NEEDED: CONSIDER TO CHECK THE VARIABLE TO dummy_object and change subsequent code lines**

In [ ]:
# TO DO: REview, leave commented or delete this cell before publicaiton


# Find the Synthesis collection
synthesis_collection = next(col for col in project.get_collections()if col.props.get("$name") == "Synthesis")

# Find our practice Object
# next() finds the first item that matches our condition. Here, we use it to find a Collection or Object with a specific name.
# If no matching Collection or Object exists, next() raises a StopIteration error. Participants should check that they have completed the previous exercises and that the names match exactly
Dummy_Object = next(
    object for object in synthesis_collection.get_objects()
    if object.props.get("$name") == "Dummy_Object"
)

# Display the selected Object
# All three datasets will be attached to this Object
print("Collection:", synthesis_collection.props.get("$name"))
print("Practice Object:", Dummy_Object.props.get("$name"))
print("Object identifier:", Dummy_Object.identifier)

#**Expected Output:** The name and identifier of the practice Object are displayed, confirming that it has been found in the Synthesis collection.

In [ ]:
# Prepare a new dataset for upload
# RAW_DATA is the dataset type used to upload data to the Data Store

raw_dataset = o.new_dataset(
    type="RAW_DATA",

    # Attach the dataset to our practice Object
    object=Dummy_Object.identifier,

    # Upload the files from the extracted folder
    file=str(extract_dir),

    # Add metadata describing the dataset
    props={
        "$name": "Parameter 1 raw",
        "notes": "Synthetic example images for demonstration."
    }
)

# Register the dataset and upload the files
# WARNING: Running this cell again may create another dataset

raw_dataset.save()

# Display the identifier of the new dataset
print("RAW_DATA dataset uploaded!")
print("Dataset identifier:", raw_dataset.permId)

#**Expected Output:** The dataset is successfully uploaded to the Data Store and its identifier is displayed.


#### Understanding the upload command

The `o.new_dataset()` method prepares a new dataset for registration in openBIS.

The parameters tell pyBIS:

- `type`: Which type of dataset we are uploading.
- `object`: Which Object the dataset belongs to.
- `file`: Which local file or folder should be uploaded.
- `props`: Which metadata should describe the dataset.

The `.save()` method registers the dataset and uploads the files.

**Expected result:** A new `RAW_DATA` dataset is attached to `Dummy_Object`.

**GUI checkpoint:** Open `Dummy_Object` and check whether the dataset and its files are visible.


In [ ]:
# Define the image we want to upload
# The image is stored in the "datasets" folder

image_path = datasets_folder / "Synthetic-image1.png"

# Check that the image file exists
if not image_path.is_file():
    raise FileNotFoundError(f"Image not found: {image_path}")

# Pepare a new dataset for the image
# ELN_PREVIEW s the Dataset Type used for this example image

preview_dataset = o.new_dataset(
    type="ELN_PREVIEW",

    # Attach the image to our practice Object "Dummy_Object"
    object=Dummy_Object.identifier,

    # Select the image file from your computer
    file=str(image_path),

    # Add descriptive metadata
    props={
        "$name": "Synthetic image1",
        "notes": "Artificial example image for demonstration."
    }
)

#  Register and upload the dataset to openBIS
# IMPORTANT: Running this cell again may create a duplicate dataset
preview_dataset.save()

# Confirm the upload and display the dataset identifier
print("Image preview uploaded!")
print("Dataset identifier:", preview_dataset.permId)

#**Expected Output:** The image is successfully uploaded to the Data Store and its identifier is displayed.

#**TO DO FOR AN: Confirm the following and correct as required. The "Synthetic-image1.png" was already uploaded with the zip file. After running the previous code cell, the same image appears 4 times in the GUI. In the GUI-Lab Notebook tutorial, the Zip file data upload was shown to demonstrate how to upload a zip file because in the GUI, it is required to "Uncompress before import"**
#**Confirm that in pyBIS is not required to uncompress the file, and if so. Lets discuss if**
#**tocomment out the code used to upload the Zip file and add an explanatory note (in GUI vs pyBIS and zip file upload is useful...)**

### YOUR TURN:
**Change the dataset type from "ELN_PREVIEW" to "ATTACHMENT" before running this cell.**

In [ ]:
## Define the README file

readme_path = datasets_folder / "README-Synthetic-image1.txt"

# Check that the file exists
if not readme_path.is_file():
    raise FileNotFoundError(f"File not found: {readme_path}")

# Select the correct dataset type
# YOUR TURN. Replace "ELN_PREVIEW" for "ATTACHMENT"
dataset_type = "ELN_PREVIEW" 

# Check your answer before uploading
# This prevents accidentally registering the wrong dataset type

if dataset_type != "ATTACHMENT":
    raise ValueError(
        "Please change dataset_type to ATTACHMENT before continuing."
    )

# Prepare the dataset
attachment_dataset = o.new_dataset(
    type=dataset_type,

    object=Dummy_Object.identifier,

    file=str(readme_path),

    props={
        "$name": "README Synthetic image1",
        "notes": "README file explaining the synthetic image."
    }
)

# Register and upload the attachment
attachment_dataset.save()

# Confirm the upload
print("README attachment uploaded!")
print("Dataset identifier:", attachment_dataset.permId)


#### GUI checkpoint — Verify your uploaded datasets

Now verify that all three datasets have been uploaded successfully.

1. Open the BAM Data Store Playground in your browser.
2. Navigate to your assigned training Space.
3. Open the `Material_Synthesis` Project.
4. Open the `Synthesis` Collection.
5. Select the practice Object `Dummy_Object`.
6. Find the datasets attached to this Object.

**Expected result:**

You should find three new datasets:

- `Parameter 1 raw` — Type `RAW_DATA` is displayed in the left-menu when you open the `Dummy_Object`.
- `Synthetic image1` — Type `ELN_PREVIEW` is displayed as image when you open the `Dummy_Object`.
- `README Synthetic image1` — Type `ATTACHMENT`is displayed in the left-menu when you open the `Dummy_Object`.

**Congratulations!** You have used pyBIS to upload research data and documentation to the Data Store.

**Think about it:** How could this workflow help you upload and organize data from multiple experiments automatically?


#**TO DO for AN:  REVIEW AND DELETE THIS CELL BEFORE PUBLICATION**

#**review, test and lets discuss how challenge 5 should be. The advantage of pyBIS vs GUI is not customizing properties selected for metadata export of objects**
#**The strongest justification for Challenge 5 is not that the GUI lacks selective export functionality. It is that pyBIS makes customized, repeatable, multi-Collection reporting easier to automate.**

#**See AI suggested new Challenge 5: test, review and finalize**

### Challenge 5 — Automate a customized metadata report using pyBIS (30 minutes)

#### Why do this?

The Data Store graphical user interface (GUI) allows you to explore, filter, and export Object metadata. You can also select which metadata properties to include in supported table exports.

However, imagine that you regularly need to prepare a report containing Experimental Steps from several Collections.

Manually navigating through Collections, selecting columns, and exporting the results can become repetitive.

With **pyBIS**, you can create a reusable workflow that:

- Retrieves Objects from multiple Collections.
- Selects only the metadata properties needed for your report.
- Combines the results into one table.
- Exports the table to a CSV file.
- Generates a new report whenever you run the code again.

**The main advantage is automation:** You define the report once and can reuse the same workflow whenever you need updated information.

#### In this exercise...

You will create a metadata report containing Experimental Steps from the `Synthesis` and `Parameter 1` Collections.

You will then change the report settings and generate a second report without rewriting the entire code.

You will perform four steps:

1. **Configure the report:** Choose which Collections and metadata properties to include.
2. **Collect metadata:** Use a Python loop to retrieve information from the selected Collections.
3. **Export the report:** Combine the information into one table and save it as a CSV file.
4. **Reuse the workflow:** Change the report settings and run the same code again.

A **loop** tells Python to repeat the same operation for multiple items.

A **pandas DataFrame** is a table organized into rows and columns.

A **CSV file** (Comma-Separated Values) is a text file used to store tabular data. It can be opened in Microsoft Excel.

#### Your task

**Part A — Generate your first report**

1. Run the configuration cell.
2. Run the report-generation cell.
3. Check that the report contains Objects from both `Synthesis` and `Parameter 1`.
4. Run the export cell to save the report as a CSV file.
5. Open the exported CSV file in VS Code or Excel.

**Part B — Reuse the workflow**

6. Return to the configuration cell.
7. Change the selected Collections from `["Synthesis", "Parameter 1"]` to `["Synthesis"]`.
8. Change the report filename to `synthesis_report.csv`.
9. Run the report-generation and export cells again.
10. Compare the two CSV reports.

#### Expected result

You will generate two reports using the same Python workflow:

| Report | Collections included | Filename |
|---|---|---|
| First report | `Synthesis` and `Parameter 1` | `experimental_steps_report.csv` |
| Second report | `Synthesis` only | `synthesis_report.csv` |

Both reports will contain the following metadata columns:

- `collection`
- `identifier`
- `name`
- `description`

The second report should contain only Objects from the `Synthesis` Collection.

#### Important notes

- **This exercise only reads information from openBIS.** It does not modify any Objects or metadata in the Data Store.
- The CSV files are separate copies of the retrieved metadata.
- Running the report-generation cell again retrieves the current metadata from openBIS.
- Running the export cell again with the same filename overwrites the existing CSV file.
- Changing an exported CSV file does not update openBIS.

**Think about it:** If you needed to generate the same report every week, how could this Python workflow save time compared with repeating the export manually?

In [ ]:
#Configure the report

# STEP 1: Choose which Collections to include in the report
# YOUR TURN (Part B):
# Change ["Synthesis", "Parameter 1"] to ["Synthesis"]

collection_names = ["Synthesis", "Parameter 1"]

# STEP 2: Choose the filename for the exported report
# YOUR TURN (Part B):
# Replace "experimental_steps_report.csv" with "synthesis_report.csv"
# Keep the .csv extension

filename = "experimental_steps_report.csv"

# STEP 3: Choose which metadata properties to include
# Each entry contains:
# - The column name to display in the report
# - The corresponding metadata property in openBIS

metadata_fields = {
    "name": "$name",
    "description": "experimental_step.experimental_description"
}

# Display the report settings

print("Selected Collections:", collection_names)
print("Selected metadata properties:", list(metadata_fields.keys()))
print("Report filename:", filename)


In [ ]:
#**Generate the Metadata Report**


# STEP 1: Import pandas
# pandas helps us organize the retrieved metadata into a table

import pandas as pd

# STEP 2: Create an empty list to store the report data
# Each Object will become one row in our report

report_rows = []

# STEP 3: Retrieve the Collections in our Project

project_collections = project.get_collections()

# STEP 4: Go through all Collections in the Project
# A loop repeats the same instructions for each Collection

for collection in project_collections:

    # Read the Collection name
    collection_name = collection.props.get("$name")

    # Only process Collections selected in our report settings
    if collection_name in collection_names:

        print("Processing Collection:", collection_name)

        # STEP 5: Retrieve all Objects in the selected Collection

        for obj in collection.get_objects():

            # STEP 6: Create a new row for this Object
            # Always include the Collection and Object identifier

            row = {
                "collection": collection_name,
                "identifier": obj.identifier
            }

            # STEP 7: Retrieve the selected metadata properties
            # This loop reads each property defined in metadata_fields

            for column_name, property_code in metadata_fields.items():
                row[column_name] = obj.props.get(property_code)

            # Add the completed row to our report
            report_rows.append(row)

# STEP 8: Create a pandas DataFrame
# The columns follow the order defined in our report settings

report_columns = ["collection", "identifier"] + list(metadata_fields.keys())

report = pd.DataFrame(report_rows, columns=report_columns)

# STEP 9: Display the combined report

display(report)

# STEP 10: Display a summary of the results

print("Total Objects retrieved:", len(report))

if report.empty:
    print("No Objects found. Check the selected Collection names.")
else:
    print("\nObjects per Collection:")
    print(report["collection"].value_counts().to_string())


In [ ]:
#**Export the report to CSV**


# STEP 1: Import Path to work with file and folder locations

from pathlib import Path

# STEP 2: Define where the report will be saved
# Path.cwd() returns the current working directory

csv_path = Path.cwd() / filename

# STEP 3: Export the report to a CSV file
# index=False avoids adding an extra index column
# IMPORTANT: This overwrites an existing file with the same name

report.to_csv(csv_path, index=False)

# STEP 4: Confirm the export
# Display the full path so we can find the file

print("Report exported successfully!")
print("Number of records:", len(report))
print("Saved to:", csv_path)



#### Check — Compare your two reports

**First report**

Run the cells using the original settings:

- Collections: `Synthesis` and `Parameter 1`
- Filename: `experimental_steps_report.csv`

Open the CSV file in VS Code or Excel.

Check that the report contains Objects from both Collections.

**Second report**

Return to the configuration cell and make two changes:

1. Replace `["Synthesis", "Parameter 1"]` with `["Synthesis"]`.
2. Replace `"experimental_steps_report.csv"` with `"synthesis_report.csv"`.

Run the report-generation and export cells again.

Open the second CSV file and compare the results.

**Expected result:**

- The first CSV contains Objects from both Collections.
- The second CSV contains only Objects from `Synthesis`.
- Both reports have the same four metadata columns.
- You generated both reports using the same Python code.

**What have you demonstrated?**

You created a reusable reporting workflow. Instead of manually exporting metadata from different Collections, you can change a few settings and generate a new report automatically.

**Important:** Rerunning the report-generation cell retrieves metadata from openBIS again. This means the report can reflect changes made to the stored metadata since the previous run.


In [ ]:
#**Verify the exported CSV**
#This final cell reads the exported report back into Python and checks that its contents match the report generated in the notebook.


# STEP 1: Read the exported CSV file
# pandas loads the saved file into a new DataFrame

saved_report = pd.read_csv(csv_path)

# STEP 2: Display the CSV contents

display(saved_report)

# STEP 3: Compare the number of rows in both tables

print("Objects in generated report:", len(report))
print("Objects in exported CSV:", len(saved_report))

# STEP 4: Check that the CSV contains the expected columns and rows

if (
    len(saved_report) == len(report)
    and list(saved_report.columns) == list(report.columns)
):
    print("Verification successful: Row counts and columns match!")
else:
    print("Verification failed: Check the exported CSV.")



#### Final checkpoint — What have you learned?

You have now used pyBIS and Python to:

- Retrieve Objects from multiple Collections.
- Select specific metadata properties for a report.
- Combine the retrieved information into one table.
- Export the report to a CSV file.
- Change the report settings and generate a second report.
- Verify that the exported CSV contains the expected rows and columns.

**Congratulations!** You have created a reusable metadata-reporting workflow using pyBIS.

**The key advantage:** Instead of manually repeating the same export steps in the Data Store GUI, you can reuse your Python code to generate reports from different Collections or retrieve updated metadata whenever needed.


**Think about it:**

- Which other Collections or metadata properties would be useful to include in your research reports?
- How could this workflow save time if you needed to generate the same report every week?
- What would happen if new Experimental Steps were added to a Collection before you ran the code again?


#**TO DO for AN: After reviewing AI generated challenge 5, clean up cells for Challenge 5**

### Challenge 5 — Automate a repetitive task (30 minutes)

#### Why do this?

The Data Store graphical user interface (GUI) allows you to explore, filter, and export Object metadata. Depending on the available table options, you can also select which metadata properties to include in an export.

However, when you need to prepare the same report repeatedly or collect information from multiple Collections, **pyBIS** offers a more flexible and automated approach.

Using Python, you can select specific metadata properties, combine information from different Collections, organize the results into a consistent table, and export them as a CSV file.

You can rerun the same workflow whenever the data changes, without manually repeating the export steps in the GUI.


#### In this exercise...

You will use pyBIS to collect metadata from experimental steps in two Collections: `Synthesis` and `Parameter_1`.

You will perform four steps:

1. **Retrieve data:** Find the experimental steps in the selected Collections.
2. **Collect metadata:** Extract the Collection name, Object identifier, Object name, and experimental description.
3. **Create a table:** Organize the information into a table using **pandas**.
4. **Export and read a CSV:** Save the table as a CSV file and open it again using Python.

A **loop** tells Python to repeat the same operation for multiple records.

A **CSV file** (Comma-Separated Values) is a simple text file that stores tabular data. It can be opened in spreadsheet software such as Excel.

#### Your task

1. Run the first code cell to retrieve metadata from the selected Collections.
2. Examine the resulting table.
3. In the export cell, change the value of `filename` to a filename of your choice, keeping the `.csv` extension.
4. Run the cell to export the table as a CSV fie.
5. Find and open the CSV file in VS Code or Excel.
6. Optionally, change a value in the CSV file and save it.
7. Run the second code cell to read the CSV file back into Python.
8. Compare the imported table with the original table.

#### Expected result

- A table containing the metadata of the experimental steps from `Synthesis` and `Parameter_1`.
- A CSV file saved in your notebook's working directory.
- The ability to open and inspect the exported CSV file.
- If you edited and saved the CSV, the modified values appear when Python reads the file again.

#### Important notes

- **This exercise only reads data from openBIS.** It does not modify any research records in the Data Store.
- The exported CSV file is a separate copy of the metadata.
- Running the export cell again with the same filename will overwrite the existing CSV file.
- pyBIS makes customized, repeatable, multi-Collection reporting easier to automate.

**Think about it:** How could this approach help you prepare reports for projects containing hundreds or thousands of experimental steps?

*WARNING:*
Running this cell again will overwrite the existing CSV file.

In [ ]:
import os # for working with directories and file paths
pwd = os.getcwd()


# Choose which Collections to include in our report
# We will collect metadata from "Synthesis" and "Parameter 1"

collection_names = ["Synthesis", "Parameter 1"]

# Retrieve the Collections available in our Project

project_collections = project.get_collections()

# Create an empty list to store our report data
# Each Object will become one row in the final table

report_rows = []

# Go through the Collections in our Project
# A "for" loop repeats the instructions for each Collection

for collection in project_collections:

    # Read the readable Collection name
    collection_name = collection.props.get("$name")

    # Only process Collections selected for our report
    if collection_name in collection_names:

        # Retrieve all Objects in the selected Collection
        # This loop processes each Object one at a time
        for obj in collection.get_objects():

            # Extract metadata from each Object
            # Each dictionary represents one row in our report

            report_rows.append({
                "collection": collection_name,
                "identifier": obj.identifier,
                "name": obj.props.get("$name"),
                "description": obj.props.get("experimental_step.experimental_description"),
            })

# Convert the collected data into a pandas DataFrame
# A DataFrame is a table with rows and columns

report = pd.DataFrame(
    report_rows,
    columns=["collection", "identifier", "name", "description"]
)

# Display the report in the notebook
display(report)

# Choose a filename for the CSV report
# YOUR TURN: Change the filename, but keep the .csv extension
filename = "my_openbis_experimental_steps.csv"

# Export the DataFrame to a CSV file
# index=False prevents pandas from adding an extra index column

report.to_csv(os.path.join(pwd, filename), index=False)

# Confirm that the file was created

print(f"Exported {len(report)} records to {filename}")

#**Expected Output:** A table ontaining metadata from Experimental Steps in the `Synthesis` and `Parameter 1` Collections.

#### Check — Open your exported CSV file

After running the previous cell, locate the CSV file in your notebook's working directory.

1. Open the file in VS Code or Excel.
2. Check that it contains the four columns: `collection`, `identifier`, `name`, and `description`.
3. Compare the Object names with those displayed in the Data Store GUI.

**Expected result:** The CSV contains the same metadata as the table displayed in your notebook.

**Optional task:** Change one description in the CSV file and save it.

In the next step, we will read the CSV file back into Python to see the saved contents.

**Remember:** Changing the CSV file does not change the metadata stored in openBIS.


**Note, this is not a task for the workshop*
**If all the data appears in one column:**

Excel may not automatically recognize the comma (`,`) as the column separator.

To import the file correctly:

1. Open a blank Excel workbook.
2. Select **Data → From Text/CSV**.
3. Choose your exported CSV file.
4. In the import preview, select **Comma** as the delimiter.
5. Click **Load**.

Your data should now appear in four separate columns.

In [ ]:
# Read the CSV file we created in the previous cell
# pandas uses read_csv() to load the file into a DataFrame

# The variable "filename" already contains the name of our CSV file
# You do not need to enter the filename again

table = pd.read_csv((os.path.join(pwd, filename)))

# Display the contents of the CSV file
# If you edited and saved the CSV, you should see your changes here

display(table)

# Display the number of rows in the CSV file

print("Number of records in the CSV:", len(table))

#*Expected output:* The number of records in the CSV file is displayed, confirming that the data was successfully read from the file.


#### Final checkpoint — What have you learned?

You have now used Python to:

- Retrieve experimental steps from multiple Collections.
- Extract metadata from openBIS Objects.
- Organize the information into a table using pandas.
- Export the table to a CSV file.
- Read the CSV file back into Python.

**Congratulations!** You have automated a repetitive research data management task using pyBIS.

**Think about it:**

- What other metadata would be useful to include in your report?
- How could this workflow save time when working with hundreds of experimental records?
- Why is it important to distinguish between a CSV export and the original data stored in openBIS?


#### Customizing the export

In the previous exercise, you created a report with the Collection, Object identifier, name, and description. Now customize that report by adding one more metadata property: the **experimental goals** for each step.

The property key for experimental goals is `experimental_step.experimental_goals`. The code below uses the same Collection-selection and Object loop as the previous exercise, adds the goals as a fifth column, and exports the customized table to an Excel workbook.


**Your task:** Run the cell and open `custom_report.csv`. Check that it contains the new `experimental_goals` column as well as the four columns from the previous report.

In [ ]:
# STEP 1: Choose which Collections to include in our report
# We will collect metadata from "Synthesis" and "Parameter 1"

collection_names = ["Synthesis", "Parameter 1"]

# STEP 2: Retrieve the Collections available in our Project

project_collections = project.get_collections()

# STEP 3: Create an empty list to store our report data
# Each Object will become one row in the final table

report_rows = []

# STEP 4: Go through the Collections in our Project
# A "for" loop repeats the instructions for each Collection

for collection in project_collections:

    # Read the human-readable Collection name
    collection_name = collection.props.get("$name")

    # Only process Collections selected for our report
    if collection_name in collection_names:

        # STEP 5: Retrieve all Objects in this Collection

        for obj in collection.get_objects():

            # STEP 6: Extract metadata from each Object
            # Each dictionary represents one row in our report

            report_rows.append({
                "collection": collection_name,
                "identifier": obj.identifier,
                "name": obj.props.get("$name"),
                "description": obj.props.get("experimental_step.experimental_description"),
                "experimental_goals": obj.props.get("experimental_step.experimental_goals")# #Additional metadata property to include in the report
            })

# STEP 7: Convert the collected data into a pandas DataFrame
# A DataFrame is a table with rows and columns

report = pd.DataFrame(
    report_rows,
    columns=["collection", "identifier", "name", "description", "experimental_goals"]
)

# STEP 8: Display the report in the notebook

display(report)

# STEP 9: Choose a filename for the CSV report
# YOUR TURN: Change the filename, but keep the .csv extension

filename = "custom_report.csv"

# STEP 10: Export the DataFrame to a CSV file
# index=False prevents pandas from adding an extra index column

report.to_csv(os.path.join(pwd, filename), index=False)

# STEP 11: Confirm that the file was created

print(f"Exported {len(report)} records to {filename}")

In [ ]:
# STEP 1: Read the CSV file we created in the previous cell
# pandas uses read_csv() to load the file into a DataFrame

# The variable "filename" already contains the name of our CSV file
# You do not need to enter the filename again

table = pd.read_csv((os.path.join(pwd, filename)))

# STEP 2: Display the contents of the CSV file
# If you edited and saved the CSV, you should see your changes here

display(table)

# STEP 3: Display the number of rows in the CSV file

print("Number of records in the CSV:", len(table))



---

### Workshop wrap-up — What can you do with pyBIS?

During this workshop, you learned how to use Python and pyBIS to interact with the BAM Data Store.

You practiced how to:

1. **Retrieve** registered Experimental Steps (Objects).
2. **Explore and filter** metadata from Collections.
3. **Update** metadata of a practice Object.
4. **Upload** research datasets and link them to an Object.
5. **Automate** customized metadata reports across multiple Collections.

You have now seen how pyBIS can complement the Data Store GUI, especially when working with multiple records or performing repetitive tasks.

**Remember:** pyBIS allows you to both read and modify data in openBIS. Always check which Objects you are working with before saving changes or uploading files.

**Congratulations on completing the workshop!**

